# 🔮 Polygon Soup to Editable Scenes
### 2D Image → Editable 3D Quad Mesh (Maya-Ready USD)

This notebook runs the full pipeline on a **free Colab T4 GPU (15GB VRAM)**:
1. **Stage 0** — Image ingestion & normalization
2. **Stage 1** — 2D segmentation (SAM 2)
3. **Stage 2** — 3D reconstruction (CRM / Unique3D)
4. **Stage 2b** — Quad retopology (Quadriflow via bpy)
5. **Stage 3** — Semantic partitioning (DINOv2 + spectral clustering)
6. **Stage 4** — USD export (Y-Up for Maya)

---
⚡ **Runtime**: Select **Runtime → Change runtime type → T4 GPU** before running.

⏱️ **First run takes ~10-15 min** (downloading models). Subsequent runs are ~3-5 min.

## 1️⃣ Check GPU & Setup Environment

In [ ]:
# Check GPU availability
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    print("⚠️ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# Clone the repository
import os
REPO_DIR = "/content/polygon-soup"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/maskees/Polygon-Soup-to-editable-scenes.git {REPO_DIR}
    print("✅ Repository cloned")
else:
    print("✅ Repository already exists")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
%%capture install_output
# Install all dependencies (captured to keep output clean)
!pip install -q trimesh[all] opencv-python-headless Pillow rembg usd-core \
    scipy matplotlib tqdm click rich pyyaml huggingface_hub \
    fast_simplification bpy==4.1.0

# Install SAM 2
!pip install -q segment-anything-2 2>/dev/null || \
    pip install -q git+https://github.com/facebookresearch/sam2.git

print("✅ All packages installed")

In [ ]:
# Verify key imports
import importlib
packages = ["torch", "trimesh", "cv2", "PIL", "scipy", "click", "rich", "yaml"]
for pkg in packages:
    try:
        importlib.import_module(pkg)
        print(f"  ✅ {pkg}")
    except ImportError:
        print(f"  ❌ {pkg} — MISSING")

# Check optional
for pkg, name in [("sam2", "SAM 2"), ("pxr", "USD"), ("bpy", "Blender"), ("rembg", "rembg")]:
    try:
        importlib.import_module(pkg)
        print(f"  ✅ {name}")
    except ImportError:
        print(f"  ⚠️ {name} — not available (optional)")

## 2️⃣ Download Model Checkpoints

First run downloads ~15-20 GB of models. They're cached in `/content/polygon-soup/checkpoints/` and persist until the Colab runtime resets.

In [ ]:
import os
import urllib.request
from pathlib import Path
from huggingface_hub import snapshot_download

CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)

# ── SAM 2 ──
sam2_path = CHECKPOINT_DIR / "sam2" / "sam2_hiera_large.pt"
sam2_path.parent.mkdir(exist_ok=True)
if not sam2_path.exists():
    print("⬇️ Downloading SAM 2 (~2.4 GB)...")
    urllib.request.urlretrieve(
        "https://dl.fbaipublicfiles.com/segment_anything_2/sam2_hiera_large.pt",
        str(sam2_path)
    )
    print(f"  ✅ SAM 2 downloaded ({sam2_path.stat().st_size / 1e9:.1f} GB)")
else:
    print(f"  ✅ SAM 2 already cached ({sam2_path.stat().st_size / 1e9:.1f} GB)")

# ── CRM ──
crm_dir = CHECKPOINT_DIR / "crm"
if not (crm_dir / "ccm-diffusion.pth").exists():
    print("⬇️ Downloading CRM (~4.5 GB)...")
    snapshot_download(repo_id="Zhengyi/CRM", local_dir=str(crm_dir))
    print("  ✅ CRM downloaded")
else:
    print("  ✅ CRM already cached")

# ── Unique3D ──
unique3d_dir = CHECKPOINT_DIR / "unique3d"
if not unique3d_dir.exists() or len(list(unique3d_dir.glob("*"))) == 0:
    print("⬇️ Downloading Unique3D (~6 GB)...")
    snapshot_download(repo_id="Wuvin/Unique3D", local_dir=str(unique3d_dir))
    print("  ✅ Unique3D downloaded")
else:
    print("  ✅ Unique3D already cached")

print("\n🎉 All checkpoints ready!")

## 3️⃣ Upload Your Image

Choose your mode:
- **Single image**: Upload 1 image (uses CRM backend)
- **4 views**: Upload front, back, left, right images (uses Unique3D backend)

In [ ]:
import shutil
from google.colab import files
from IPython.display import display, HTML
from PIL import Image
import ipywidgets as widgets

# Setup directories
INPUT_DIR = Path("data/input/colab_session")
OUTPUT_DIR = Path("data/output/colab_session")

if INPUT_DIR.exists():
    shutil.rmtree(INPUT_DIR)
INPUT_DIR.mkdir(parents=True, exist_ok=True)

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Mode selection
display(HTML("<h3>📤 Upload Mode</h3>"))
mode_selector = widgets.RadioButtons(
    options=["Single Image (CRM)", "4 Views (Unique3D)"],
    value="Single Image (CRM)",
    description="Mode:",
)
display(mode_selector)

print("\n👆 Select mode above, then run the next cell to upload.")

In [ ]:
# Upload images based on selected mode
mode = mode_selector.value

if "Single" in mode:
    print("📤 Upload a single image:")
    uploaded = files.upload()
    if uploaded:
        filename = list(uploaded.keys())[0]
        # Save as front.png and duplicate for other views
        img = Image.open(filename).convert("RGB")
        img.save(str(INPUT_DIR / "front.png"))
        for view in ["back.png", "left.png", "right.png"]:
            shutil.copy(INPUT_DIR / "front.png", INPUT_DIR / view)
        BACKEND = "crm"
        print(f"\n✅ Saved as front.png ({img.size[0]}×{img.size[1]})")
        display(img.resize((256, 256)))
else:
    BACKEND = "unique3d"
    for view_name in ["front", "back", "left", "right"]:
        print(f"\n📤 Upload {view_name.upper()} view:")
        uploaded = files.upload()
        if uploaded:
            filename = list(uploaded.keys())[0]
            img = Image.open(filename).convert("RGB")
            img.save(str(INPUT_DIR / f"{view_name}.png"))
            print(f"  ✅ {view_name}.png ({img.size[0]}×{img.size[1]})")

# Verify all files exist
print("\n📁 Input directory:")
for f in sorted(INPUT_DIR.iterdir()):
    print(f"  {f.name} ({f.stat().st_size / 1024:.0f} KB)")

print(f"\n🔧 Backend: {BACKEND}")

## 4️⃣ Run the Pipeline

This runs all 6 stages. On a T4 GPU with `--low-vram`:
- Stage 0 (Ingest): ~2s
- Stage 1 (SAM 2): ~10-30s
- Stage 2 (CRM/Unique3D): ~60-180s
- Stage 2b (Quad Retopology): ~30-60s
- Stage 3 (Partitioning): ~30-90s
- Stage 4 (USD Export): ~5s

In [ ]:
import subprocess, time

print("🚀 Running full pipeline...")
print("=" * 60)
start = time.time()

cmd = [
    "python", "main.py",
    "--input", str(INPUT_DIR),
    "--output", str(OUTPUT_DIR),
    "--backend", BACKEND,
    "--low-vram",
    "--verbose",
]

process = subprocess.run(
    cmd,
    capture_output=False,  # Show output live
    text=True,
    timeout=3600,  # 1 hour max
)

elapsed = time.time() - start
print("=" * 60)

if process.returncode == 0:
    print(f"\n🎉 Pipeline completed in {elapsed:.0f}s ({elapsed/60:.1f} min)")
else:
    print(f"\n❌ Pipeline failed after {elapsed:.0f}s")
    print("Check the output above for error details.")

## 5️⃣ View & Download Results

In [ ]:
# List all output files
from pathlib import Path
from IPython.display import display, HTML

print("📦 Output files:")
print("=" * 60)

output_files = []
for f in sorted(OUTPUT_DIR.rglob("*")):
    if f.is_file():
        size_mb = f.stat().st_size / (1024 * 1024)
        rel = f.relative_to(OUTPUT_DIR)
        print(f"  {rel}  ({size_mb:.1f} MB)")
        output_files.append(f)

# Show intermediate results
print("\n🖼️ Intermediate stage outputs:")
for stage_dir in ["stage0_ingest", "stage1_segment", "stage2_reconstruct", 
                  "stage2b_retopologize", "stage3_partition"]:
    d = OUTPUT_DIR / "intermediate" / stage_dir
    if d.exists():
        n_files = len(list(d.iterdir()))
        print(f"  ✅ {stage_dir}: {n_files} file(s)")
    else:
        print(f"  ⬜ {stage_dir}: not generated")

In [ ]:
# Visualize the 3D mesh inline (if trimesh is available)
import trimesh
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import numpy as np

# Try to load and display the final mesh
mesh_candidates = [
    OUTPUT_DIR / "intermediate" / "stage2b_retopologize" / "monolithic_quad_mesh.obj",
    OUTPUT_DIR / "intermediate" / "stage2_reconstruct" / "monolithic_mesh.obj",
]

mesh_path = None
for candidate in mesh_candidates:
    if candidate.exists():
        mesh_path = candidate
        break

if mesh_path:
    mesh = trimesh.load(str(mesh_path), force="mesh")
    print(f"📐 Mesh: {len(mesh.vertices)} vertices, {len(mesh.faces)} faces")
    print(f"   Watertight: {mesh.is_watertight}")
    print(f"   Bounds: {mesh.bounds.tolist()}")
    
    # Simple 3D plot
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    # Subsample faces for visualization
    max_display_faces = 5000
    if len(mesh.faces) > max_display_faces:
        indices = np.random.choice(len(mesh.faces), max_display_faces, replace=False)
        display_faces = mesh.faces[indices]
    else:
        display_faces = mesh.faces
    
    verts = mesh.vertices
    polys = [[verts[vi] for vi in face] for face in display_faces]
    collection = Poly3DCollection(polys, alpha=0.6, edgecolor='#333', linewidth=0.1)
    collection.set_facecolor('#4a90d9')
    ax.add_collection3d(collection)
    
    scale = mesh.extents.max()
    center = mesh.centroid
    ax.set_xlim(center[0] - scale/2, center[0] + scale/2)
    ax.set_ylim(center[1] - scale/2, center[1] + scale/2)
    ax.set_zlim(center[2] - scale/2, center[2] + scale/2)
    ax.set_title(f"Reconstructed Mesh ({len(mesh.faces)} faces)", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ No mesh file found to visualize")

In [ ]:
# Show segmented parts (if Stage 3 completed)
parts_dir = OUTPUT_DIR / "intermediate" / "stage3_partition"
if parts_dir.exists():
    part_files = sorted(parts_dir.glob("part_*.obj"))
    print(f"🧩 {len(part_files)} semantic parts:")
    
    for pf in part_files:
        pm = trimesh.load(str(pf), force="mesh")
        print(f"  {pf.name}: {len(pm.vertices)} verts, {len(pm.faces)} faces")
        
        # Check if quads
        with open(pf) as f:
            quad_count = sum(1 for line in f if line.startswith("f ") and len(line.split()) == 5)
        tri_count = len(pm.faces)
        if quad_count > 0:
            print(f"    → ✅ {quad_count} quads (Maya-ready!)")
else:
    print("⚠️ Stage 3 output not found")

In [ ]:
# Download the final USD file(s) and sub-meshes
from google.colab import files
import zipfile

# Create a zip of all outputs
zip_path = "/content/reconstruction_output.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in OUTPUT_DIR.rglob("*"):
        if f.is_file():
            arcname = f.relative_to(OUTPUT_DIR)
            zf.write(f, arcname)
            
zip_size = Path(zip_path).stat().st_size / (1024 * 1024)
print(f"📦 Output zip: {zip_size:.1f} MB")
print("\n⬇️ Download starting...")
files.download(zip_path)

## 📋 Importing in Maya

After downloading `reconstruction_output.zip`:

1. Extract the zip
2. Open Maya
3. **File → Import** → select `usd/scene_y_up.usda`
4. Each semantic part appears as a separate mesh in the Outliner
5. Parts have **quad topology** — fully editable with Maya's polygon tools
6. Toggle part visibility via the Outliner

### Alternative: Import individual OBJ parts
The `intermediate/stage3_partition/` folder contains individual `.obj` files:
- `part_000.obj`, `part_001.obj`, etc.
- Each preserves quad topology from the retopology stage
- Import individually via **File → Import** in Maya

---
## 🔄 Run Again with Different Settings

Go back to **Section 3** to upload a new image, or modify the pipeline command in **Section 4**.

**Useful flags:**
- `--low-vram` — Use 6GB GPU mode (enabled by default on Colab)
- `--backend crm` — Single-image reconstruction
- `--backend unique3d` — Multi-view reconstruction (better quality)
- `--stages 0,1,2` — Run only specific stages
- `--skip-existing` — Skip stages whose outputs already exist